# 黑白照片上色 - GPU 训练 Notebook

本 Notebook 用于在 GPU 平台（如 AutoDL、Colab）上训练图像上色模型。

## 使用步骤
1. 上传项目代码到 GPU 平台
2. 按顺序运行下面的代码块
3. 监控训练进度

## 1. 环境检查

In [ ]:
import sys
import os
import torch

print("=" * 50)
print("环境信息")
print("=" * 50)
print(f"Python 版本: {sys.version}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 警告：CUDA 不可用，将使用 CPU 训练（非常慢）")

## 2. 设置项目路径

In [ ]:
# 设置项目根目录
PROJECT_ROOT = "/root"  # 根据实际情况修改
PROJECT_DIR = os.path.join(PROJECT_ROOT, "colorization_project")

# 添加到 Python 路径
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# 切换到项目目录
os.chdir(PROJECT_DIR)
print(f"当前目录: {os.getcwd()}")

# 检查关键文件
required_files = ['train.py', 'test.py', 'requirements.txt']
for file in required_files:
    if os.path.exists(file):
        print(f"✓ {file}")
    else:
        print(f"✗ {file} 未找到")

## 3. 安装依赖（如果需要）

In [ ]:
# 如果依赖未安装，运行此单元格
!pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

## 4. 下载数据集（如果需要）

In [ ]:
# 检查数据集是否存在
DATASET_DIR = os.path.join(PROJECT_ROOT, "datasets/coco")

if os.path.exists(os.path.join(DATASET_DIR, "train2017")):
    print("✓ COCO 数据集已存在")
    train_count = len(os.listdir(os.path.join(DATASET_DIR, "train2017")))
    print(f"  训练图像数量: {train_count}")
else:
    print("⚠️ COCO 数据集未找到")
    print("开始下载...（这可能需要一些时间）")
    !python -m data.download_data --dataset coco --data_root {PROJECT_ROOT}/datasets

## 5. 导入项目模块

In [ ]:
from colorization_project.data.dataset import COCOColorizationDataset
from colorization_project.models.colorization_net import ColorizationNet
from colorization_project.training.config import TrainingConfig
from colorization_project.training.trainer import Trainer

print("✓ 所有模块导入成功")

## 6. 配置训练参数

In [ ]:
# 训练配置
config = TrainingConfig(
    # 数据相关
    data_root=os.path.join(PROJECT_ROOT, "datasets"),
    dataset_type="coco",
    image_size=256,
    crop_size=224,
    batch_size=32,  # 根据 GPU 显存调整
    num_workers=4,
    
    # 训练相关
    num_epochs=50,
    learning_rate=1e-4,
    weight_decay=1e-4,
    warmup_epochs=5,
    
    # 优化
    use_amp=True,  # 混合精度训练
    gradient_clip=1.0,
    
    # 保存和日志
    checkpoint_dir=os.path.join(PROJECT_ROOT, "checkpoints"),
    log_dir=os.path.join(PROJECT_ROOT, "outputs/logs"),
    save_freq=5,
    log_freq=100,
    
    # 设备
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print("训练配置:")
print(f"  数据集: {config.dataset_type}")
print(f"  批次大小: {config.batch_size}")
print(f"  训练轮数: {config.num_epochs}")
print(f"  学习率: {config.learning_rate}")
print(f"  混合精度: {config.use_amp}")
print(f"  设备: {config.device}")

## 7. 加载数据集

In [ ]:
print("加载数据集...")

# 训练集
train_dataset = COCOColorizationDataset(
    coco_root=os.path.join(config.data_root, 'coco'),
    split='train2017',
    image_size=config.image_size,
    crop_size=config.crop_size,
)

# 验证集
val_dataset = COCOColorizationDataset(
    coco_root=os.path.join(config.data_root, 'coco'),
    split='val2017',
    image_size=config.image_size,
    crop_size=config.crop_size,
)

print(f"✓ 训练集大小: {len(train_dataset)}")
print(f"✓ 验证集大小: {len(val_dataset)}")

# 测试数据加载
print("\n测试数据加载...")
l_channel, ab_channels = train_dataset[0]
print(f"  L 通道形状: {l_channel.shape}")
print(f"  ab 通道形状: {ab_channels.shape}")
print("✓ 数据加载正常")

## 8. 创建训练器

In [ ]:
print("创建训练器...")
trainer = Trainer(config, train_dataset, val_dataset)
print("✓ 训练器创建成功")
print(f"  模型参数量: {sum(p.numel() for p in trainer.model.parameters()):,}")

## 9. 开始训练

⚠️ **注意**：训练将需要较长时间（2-3 天），请确保：
1. GPU 平台不会自动断开
2. 有足够的磁盘空间保存检查点
3. 定期检查训练进度

In [ ]:
# 开始训练
print("=" * 50)
print("开始训练")
print("=" * 50)

trainer.train()

print("\n" + "=" * 50)
print("训练完成！")
print("=" * 50)

## 10. 查看训练结果

In [ ]:
# 列出保存的检查点
checkpoint_dir = config.checkpoint_dir
if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')]
    print(f"保存的检查点 ({len(checkpoints)} 个):")
    for ckpt in sorted(checkpoints):
        size = os.path.getsize(os.path.join(checkpoint_dir, ckpt)) / 1024**2
        print(f"  {ckpt} ({size:.1f} MB)")
else:
    print("未找到检查点目录")

## 11. 启动 TensorBoard（可选）

In [ ]:
# 在 Jupyter 中加载 TensorBoard
%load_ext tensorboard
%tensorboard --logdir {config.log_dir}

## 12. 测试模型（训练完成后）

In [ ]:
# 加载最佳模型
best_model_path = os.path.join(checkpoint_dir, "best_model.pth")

if os.path.exists(best_model_path):
    print(f"加载模型: {best_model_path}")
    
    # 创建模型
    model = ColorizationNet().to(config.device)
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print("✓ 模型加载成功")
    
    # 测试推理
    with torch.no_grad():
        l_test, ab_test = val_dataset[0]
        l_test = l_test.unsqueeze(0).to(config.device)
        pred_ab = model(l_test)
        print(f"  输入形状: {l_test.shape}")
        print(f"  输出形状: {pred_ab.shape}")
        print("✓ 推理测试成功")
else:
    print("未找到最佳模型文件")

## 完成

训练完成后，记得：
1. 下载检查点文件到本地
2. 下载训练日志
3. 保存 TensorBoard 数据

下载命令（在本地 Mac 上运行）：
```bash
scp user@gpu-server:/root/checkpoints/best_model.pth ~/Desktop/CV/checkpoints/
scp -r user@gpu-server:/root/outputs/logs/ ~/Desktop/CV/outputs/
```